In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
# Quality check notebook for the augmented dataset.
# No GPU required.

import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False
    print('Running locally — skipping Drive mount')

sys.path.insert(0, f'{PROJECT_DIR}/src')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'numpy', 'matplotlib'
], check=True)

print(f'PROJECT_DIR = {PROJECT_DIR}')
print('Setup complete.')

In [ ]:
# ── Cell 2: Load Augmented Data ──────────────────────────────────────────────
import json, pandas as pd
import sys
sys.path.insert(0, f'{PROJECT_DIR}/src')

aug_path = f'{PROJECT_DIR}/data/augmented/augmented_train.json'
with open(aug_path) as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f'Total records: {len(df)}')
print()
print('Columns:', df.columns.tolist())
print()
print('Label distribution:')
print(df['label'].value_counts())
print()
print('Perturbation types:')
print(df['perturbation_type'].value_counts(dropna=False))


In [ ]:
# ── Cell 3: Show 3 (Original, Perturbed) Pairs per Perturbation Type ─────────

import pandas as pd

# The augmented dataset should contain both True (original) and False (perturbed)
# records. Each False record should link back to its original via an 'original_id'
# or 'source_id' field. Adjust the column names below if needed.
ID_COL      = 'id'          # unique id for each record
ORIG_ID_COL = 'original_id' # ID of the source True record (for False records)
PERT_COL    = 'perturbation_type'

if PERT_COL not in df.columns:
    print(f'Column "{PERT_COL}" not found. Available columns: {list(df.columns)}')
else:
    perturbation_types = df[df['label_int'] == 0][PERT_COL].dropna().unique()
    print(f'Perturbation types found: {list(perturbation_types)}')
    print()

    for ptype in sorted(perturbation_types):
        print('=' * 70)
        print(f'PERTURBATION TYPE: {ptype}')
        print('=' * 70)

        false_samples = df[
            (df['label_int'] == 0) & (df[PERT_COL] == ptype)
        ].head(3)

        for i, (_, row) in enumerate(false_samples.iterrows(), 1):
            print(f'\n  Pair {i}:')

            # Try to retrieve the original
            orig_text = None
            if ORIG_ID_COL in df.columns and ID_COL in df.columns:
                orig_rows = df[
                    (df[ID_COL] == row.get(ORIG_ID_COL)) & (df['label_int'] == 1)
                ]
                if not orig_rows.empty:
                    orig_text = orig_rows.iloc[0]['text']

            if orig_text:
                print(f'  [ORIGINAL]  : {orig_text[:250]}...')
            else:
                print(f'  [ORIGINAL]  : (original not linked — check ID columns)')

            print(f'  [PERTURBED] : {row["text"][:250]}...')

        print()

In [ ]:
# ── Cell 4: Label Distribution and Class Balance ─────────────────────────────

import pandas as pd
import numpy as np

print('=== Label Distribution ===')
label_counts = df['label_int'].value_counts().sort_index()
label_pct    = (label_counts / len(df) * 100).round(2)

for lbl, cnt, pct in zip(label_counts.index, label_counts.values, label_pct.values):
    lname = 'True (original)' if lbl == 1 else 'False (misinformation)'
    print(f'  label={lbl} ({lname:25s}) : {cnt:5d}  ({pct:.1f}%)')

balance_ratio = label_counts.min() / label_counts.max()
print(f'\nBalance ratio (min/max) : {balance_ratio:.4f}')
if balance_ratio >= 0.9:
    print('Class balance: GOOD (ratio >= 0.9)')
elif balance_ratio >= 0.7:
    print('Class balance: ACCEPTABLE (ratio 0.7–0.9)')
else:
    print('Class balance: IMBALANCED (ratio < 0.7) — consider resampling')
print()

if 'perturbation_type' in df.columns:
    print('=== False-Class Breakdown by Perturbation Type ===')
    pt_counts = df[df['label_int'] == 0]['perturbation_type'].value_counts()
    for ptype, cnt in pt_counts.items():
        print(f'  {ptype:35s} : {cnt}')
    print(f'  TOTAL (False)                          : {pt_counts.sum()}')
    print()

if 'split' in df.columns:
    print('=== Label Distribution by Source Split ===')
    print(df.groupby(['split', 'label_int']).size().unstack(fill_value=0))

In [ ]:
# ── Cell 5: Text Length Distribution (Original vs. Augmented) ────────────────

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Separate original True-class from perturbed False-class
orig_lengths = df[df['label_int'] == 1]['text_len']
pert_lengths = df[df['label_int'] == 0]['text_len']

print('Text length statistics (characters):')
stats = pd.DataFrame({
    'Original (True)':   orig_lengths.describe(),
    'Augmented (False)': pert_lengths.describe(),
})
print(stats.round(1))
print()

# Check for anomalies: very short or very long augmented texts
SHORT_THRESH = 50
LONG_THRESH  = 2000

short_orig = (orig_lengths < SHORT_THRESH).sum()
short_pert = (pert_lengths < SHORT_THRESH).sum()
long_orig  = (orig_lengths > LONG_THRESH).sum()
long_pert  = (pert_lengths > LONG_THRESH).sum()

print(f'Suspiciously short (< {SHORT_THRESH} chars):')
print(f'  Original  : {short_orig}')
print(f'  Augmented : {short_pert}')
print(f'Suspiciously long (> {LONG_THRESH} chars):')
print(f'  Original  : {long_orig}')
print(f'  Augmented : {long_pert}')
print()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(orig_lengths, bins=40, alpha=0.6, color='steelblue',
             label=f'Original (True, n={len(orig_lengths)})')
axes[0].hist(pert_lengths, bins=40, alpha=0.6, color='tomato',
             label=f'Augmented (False, n={len(pert_lengths)})')
axes[0].set_xlabel('Text length (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Text Length Distribution')
axes[0].legend()

# Box plot for easier comparison
axes[1].boxplot(
    [orig_lengths.values, pert_lengths.values],
    labels=['Original (True)', 'Augmented (False)'],
    patch_artist=True,
    boxprops=dict(facecolor='lightblue'),
)
axes[1].set_ylabel('Text length (chars)')
axes[1].set_title('Text Length Box Plot')

plt.suptitle('Text Length: Original vs. Augmented', fontsize=13)
plt.tight_layout()

plot_path = f'{PROJECT_DIR}/results/text_length_distribution.png'
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')

# Perturbation-type breakdown of lengths
if 'perturbation_type' in df.columns:
    print('\nMean text length by perturbation type:')
    pt_lengths = df[df['label_int'] == 0].groupby('perturbation_type')['text_len'].agg(['mean', 'std'])
    print(pt_lengths.round(1))